# 🌽 CornSight — Improved Training Run

Same idea as the test notebook, but bigger:
- **4× more training data**: the full official training set (3,083 image chips) instead of ~600.
- **Longer training**: 12 epochs instead of 5, with a learning rate that slowly decreases (helps the model settle on a better answer).
- **Publishes the results to GitHub** (public repo [`AlonsoManchego/cornsight`](https://github.com/AlonsoManchego/cornsight), folder `training_outputs/`), where Notebook 2 reads them.
- **Scores every test image** and saves a results table (`chip_results.csv`): location, photo dates, predicted vs. real corn share, and corn health (NDVI). That table is what goes into MongoDB next.

**Before you start:** `Runtime → Change runtime type → T4 GPU → Save`, then run top to bottom.

⏱️ Expected time: **~50–60 min** (download + unzip ~10–15 min, training ~40 min). The progress bar in Step 7 shows the remaining time. If it's too slow, lower `EPOCHS` in Step 3.

If any cell shows a red error, copy the **whole error message** and send it back.

## Step 1 — Install the toolkit
⚠️ If Colab shows a **"Restart session"** warning after this cell, click it, then **skip this cell** and continue from Step 2.

In [ ]:
!pip install -q terratorch==1.2.14

## Step 2 — Load libraries

In [ ]:
import os, glob, shutil, subprocess, time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch

import albumentations as A
from albumentations.pytorch import ToTensorV2
import lightning.pytorch as pl
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from terratorch.datamodules import MultiTemporalCropClassificationDataModule
from terratorch.tasks import SemanticSegmentationTask
from terratorch.datasets.transforms import FlattenTemporalIntoChannels, UnflattenTemporalFromChannels

HAS_GPU = torch.cuda.is_available()
print("GPU:", torch.cuda.get_device_name(0) if HAS_GPU else "❌ NONE — go to Runtime → Change runtime type → T4 GPU")

## Step 3 — Settings

In [ ]:
DATA_ROOT = "/content/crop_data" if os.path.exists("/content") else "./crop_data"  # images stay on the fast local disk
SAVE_DIR = "/content/cornsight_results" if os.path.exists("/content") else "./cornsight_results"  # results, published to GitHub in Step 13
os.makedirs(SAVE_DIR, exist_ok=True)

GITHUB_REPO = "AlonsoManchego/cornsight"     # public repo that Notebook 2 reads from
GITHUB_FOLDER = "training_outputs"

# --- Model size ---
# "prithvi_eo_v2_tiny_tl" (~5M params, fast)  or  "prithvi_eo_v2_100_tl" (~100M params, slower, more accurate)
BACKBONE = "prithvi_eo_v2_tiny_tl"

EPOCHS      = 12     # ~40 min on a T4. Raise to 20 if you have time, lower to 8 if in a hurry
BATCH_SIZE  = 16
LR          = 2.0e-4
NUM_WORKERS = 2      # Colab has 2 CPU cores
SEED        = 0

BANDS = ["BLUE", "GREEN", "RED", "NIR_NARROW", "SWIR_1", "SWIR_2"]
NUM_FRAMES = 3

CLASS_NAMES = ["Natural_Vegetation", "Forest", "Corn", "Soybeans", "Wetlands", "Developed_Barren",
               "Open_Water", "Winter_Wheat", "Alfalfa", "Fallow_Idle_Cropland", "Cotton", "Sorghum", "Other"]
CORN = CLASS_NAMES.index("Corn")
CLASS_WEIGHTS = [0.386375, 0.661126, 0.548184, 0.640482, 0.876862, 0.925186, 3.249462,
                 1.542289, 2.175141, 2.272419, 3.062762, 3.626097, 1.198702]

## Step 4 — Download the full dataset (~11.8 GB, ~10–15 min)
- **Training set:** 3,083 chips. The model learns from these.
- **Validation set:** 771 chips. Used to test the model on places it has never seen.

Each chip is a 224×224-pixel square (~6.7 km wide) somewhere in the US, photographed 3 times during the 2022 growing season, with a label for the crop on every pixel.

In [ ]:
from huggingface_hub import hf_hub_download

REPO = "ibm-nasa-geospatial/multi-temporal-crop-classification"
os.makedirs(DATA_ROOT, exist_ok=True)

def fetch_and_extract(archive):
    folder = os.path.join(DATA_ROOT, archive.replace(".tgz", ""))
    if os.path.isdir(folder) and len(os.listdir(folder)) > 0:
        print(f"✅ {folder} already there, skipping")
        return
    t0 = time.time()
    path = hf_hub_download(REPO, archive, repo_type="dataset", local_dir=DATA_ROOT)
    print(f"Downloaded {archive} in {time.time()-t0:.0f}s, unzipping...")
    # --exclude skips hidden macOS junk files ('._chip...') that would crash the loader
    subprocess.run(["tar", "-xzf", path, "-C", DATA_ROOT, "--exclude=._*", "--warning=no-unknown-keyword"], check=True)
    os.remove(path)  # free disk space
    print(f"✅ {archive} ready in {time.time()-t0:.0f}s")

for f in ["chips_df.csv", "training_data.txt", "validation_data.txt"]:
    hf_hub_download(REPO, f, repo_type="dataset", local_dir=DATA_ROOT)
for archive in ["validation_chips.tgz", "training_chips.tgz"]:
    fetch_and_extract(archive)

print("Training chips:  ", len(glob.glob(os.path.join(DATA_ROOT, "training_chips", "*_merged.tif"))))
print("Validation chips:", len(glob.glob(os.path.join(DATA_ROOT, "validation_chips", "*_merged.tif"))))

## Step 5 — Prepare the data

In [ ]:
train_transforms = [
    FlattenTemporalIntoChannels(),
    A.HorizontalFlip(),
    A.VerticalFlip(),
    ToTensorV2(),
    UnflattenTemporalFromChannels(n_timesteps=NUM_FRAMES),
]

data_module = MultiTemporalCropClassificationDataModule(
    data_root=DATA_ROOT,
    batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,
    train_transform=train_transforms,
    expand_temporal_dimension=True,
    reduce_zero_label=True,
    use_metadata=True,     # gives the model each chip's location + photo dates
)
data_module.setup("fit")
print("Training chips:  ", len(data_module.train_dataset))
print("Validation chips:", len(data_module.val_dataset))

## Step 6 — Build the model
Pre-trained Prithvi backbone (already "understands" satellite images) + a decoder that labels every pixel.

In [ ]:
pl.seed_everything(SEED)

model_args = dict(
    backbone=BACKBONE,
    backbone_pretrained=True,
    backbone_bands=BANDS,
    backbone_num_frames=NUM_FRAMES,
    backbone_coords_encoding=["time", "location"],
    necks=[
        dict(name="SelectIndices", indices=[2, 5, 8, 11]),   # tap 4 of the 12 transformer layers (same for tiny and 100M)
        dict(name="ReshapeTokensToImage", effective_time_dim=NUM_FRAMES),
    ],
    decoder="UperNetDecoder",
    decoder_channels=128 if "tiny" in BACKBONE else 256,
    decoder_scale_modules=True,
    num_classes=len(CLASS_NAMES),
    head_dropout=0.1,
    rescale=True,
)

model = SemanticSegmentationTask(
    model_args=model_args,
    model_factory="EncoderDecoderFactory",
    loss="ce",
    class_weights=CLASS_WEIGHTS,
    class_names=CLASS_NAMES,
    ignore_index=-1,
    lr=LR,
    optimizer="AdamW",
    optimizer_hparams=dict(weight_decay=0.1),
    scheduler="CosineAnnealingLR",               # learning rate slowly decreases to zero
    scheduler_hparams=dict(T_max=EPOCHS),
    freeze_backbone=False,
    plot_on_val=False,
)
print(f"✅ {BACKBONE} built — {sum(p.numel() for p in model.parameters())/1e6:.1f}M parameters")

## Step 7 — Train
The best version of the model (highest validation score) is saved after every epoch and used for the final scores. The model file is ~113 MB, over GitHub's 100 MB limit, so it is **not** published; download it from the Colab Files panel (`cornsight_results/checkpoints/`) if you want to keep it.

In [ ]:
checkpoint_callback = ModelCheckpoint(
    dirpath=os.path.join(SAVE_DIR, "checkpoints"),
    monitor="val/mIoU", mode="max", save_top_k=1,
    filename=f"{BACKBONE}-best-{{epoch:02d}}",
)
logger = CSVLogger(SAVE_DIR, name="logs")

trainer = pl.Trainer(
    accelerator="auto",
    devices=1,
    precision="16-mixed" if HAS_GPU else "32",   # T4 GPUs don't support bf16
    max_epochs=EPOCHS,
    logger=logger,
    callbacks=[checkpoint_callback],
    log_every_n_steps=10,
)

t0 = time.time()
trainer.fit(model, datamodule=data_module)
TRAIN_MIN = (time.time() - t0) / 60
print(f"✅ Training finished in {TRAIN_MIN:.1f} min")
print("Best checkpoint:", checkpoint_callback.best_model_path)

## Step 8 — Learning curve
Shows the model getting better as it trains. Good chart for the slides.

In [ ]:
m = pd.read_csv(os.path.join(logger.log_dir, "metrics.csv"))
val = m.dropna(subset=["val/mIoU"]).groupby("epoch").last()

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(val.index + 1, val["val/mIoU"], marker="o", label="All land types (mIoU)")
if "val/IoU_Corn" in val:
    ax[0].plot(val.index + 1, val["val/IoU_Corn"], marker="o", color="#e8b100", label="Corn IoU")
ax[0].set_xlabel("Epoch"); ax[0].set_ylabel("IoU (higher is better)")
ax[0].set_title("Accuracy on unseen images"); ax[0].legend()

if "train/loss" in m:
    tr = m.dropna(subset=["train/loss"]).groupby("epoch")["train/loss"].mean()
    ax[1].plot(tr.index + 1, tr.values, marker="o", label="Training")
if "val/loss" in val:
    ax[1].plot(val.index + 1, val["val/loss"], marker="o", label="Validation")
ax[1].set_xlabel("Epoch"); ax[1].set_ylabel("Error (loss, lower is better)")
ax[1].set_title("Model error during training"); ax[1].legend()
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "learning_curve.png"), dpi=150); plt.show()

## Step 9 — Final scores
For reference: test notebook got **0.357** corn IoU; IBM's fully-trained (larger) model got **0.55**.

In [ ]:
ckpt_path = checkpoint_callback.best_model_path or None
results = trainer.test(model, datamodule=data_module, ckpt_path=ckpt_path)[0]

print("\n===== RESULTS =====")
for k in ["test/mIoU", "test/Pixel_Accuracy", "test/IoU_Corn", "test/IoU_Soybeans"]:
    if k in results:
        print(f"{k:25s} {results[k]:.3f}")

iou = {k.replace("test/IoU_", ""): v for k, v in results.items() if k.startswith("test/IoU_")}
names, vals = zip(*sorted(iou.items(), key=lambda kv: kv[1]))
plt.figure(figsize=(8, 5))
plt.barh(names, vals, color=["#e8b100" if n == "Corn" else "#8aa2b8" for n in names])
plt.axvline(0.357, color="gray", ls="--", lw=1); plt.text(0.36, 0.2, "test notebook corn", fontsize=8, color="gray")
plt.xlabel("IoU (higher is better)"); plt.title("How well the model recognizes each land type")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "iou_by_class.png"), dpi=150); plt.show()

pd.Series({**results, "backbone": BACKBONE, "epochs": EPOCHS, "train_minutes": round(TRAIN_MIN, 1)}) \
  .to_csv(os.path.join(SAVE_DIR, "test_scores.csv"))

## Step 10 — Score every validation image → `chip_results.csv`
For each of the 771 chips: where it is, when it was photographed, how much is corn (predicted vs. real), and how healthy the corn looks on each date (NDVI). This table feeds the next steps (MongoDB, USDA comparison, the AI agent).

In [ ]:
def unwrap(p):
    """Predictions come back nested as ((prediction, label), filenames). Dig out the prediction tensor."""
    while isinstance(p, (tuple, list)):
        p = p[0]
    return p

preds = trainer.predict(model, datamodule=data_module, ckpt_path=ckpt_path)
pred_masks = torch.cat([unwrap(p).cpu() for p in preds])          # (chips, H, W), same order as the dataset
ds = data_module.predict_dataset
meta = pd.read_csv(os.path.join(DATA_ROOT, "chips_df.csv")).set_index("chip_id")
R, NIR = BANDS.index("RED"), BANDS.index("NIR_NARROW")

rows, i = [], 0
for batch in trainer.predict_dataloaders:                         # same images, same order, unprocessed
    img, mask = batch["image"].float(), batch["mask"]
    ndvi = (img[:, NIR] - img[:, R]) / (img[:, NIR] + img[:, R] + 1e-6)   # (b, time, H, W)
    for b in range(img.shape[0]):
        chip_id = os.path.basename(ds.image_files[i]).replace("_merged.tif", "")
        pm, valid = pred_masks[i], mask[b] >= 0
        corn_pred, corn_true = (pm == CORN), (mask[b] == CORN)
        row = {
            "chip_id": chip_id,
            "lat": round(float(batch["location_coords"][b][0]), 5),
            "lon": round(float(batch["location_coords"][b][1]), 5),
            "tile": meta.loc[chip_id, "tile"],
            "date_1": meta.loc[chip_id, "first_img_date"][:10],
            "date_2": meta.loc[chip_id, "middle_img_date"][:10],
            "date_3": meta.loc[chip_id, "last_img_date"][:10],
            "corn_share_pred": round(corn_pred.float().mean().item(), 4),
            "corn_share_true": round((corn_true & valid).float().sum().item() / max(valid.sum().item(), 1), 4),
        }
        for t in range(NUM_FRAMES):
            row[f"corn_ndvi_{t+1}"] = round(ndvi[b, t][corn_pred].mean().item(), 4) if corn_pred.sum() > 50 else None
        rows.append(row)
        i += 1

chips = pd.DataFrame(rows)
chips.to_csv(os.path.join(SAVE_DIR, "chip_results.csv"), index=False)
print(f"✅ Saved {len(chips)} chips to {SAVE_DIR}/chip_results.csv")
chips.head()

## Step 11 — Charts for the presentation
1. **Does the model measure corn acreage correctly?** Predicted vs. real corn share per chip. Dots on the diagonal = perfect.
2. **Where is the corn?** Each chip on a US map, colored by predicted corn share. The Corn Belt should light up.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(13, 5))

ax[0].scatter(chips.corn_share_true * 100, chips.corn_share_pred * 100, s=10, alpha=0.5, color="#e8b100")
ax[0].plot([0, 100], [0, 100], "k--", lw=1)
r = chips[["corn_share_true", "corn_share_pred"]].corr().iloc[0, 1]
ax[0].set_xlabel("Real corn share (%)"); ax[0].set_ylabel("Predicted corn share (%)")
ax[0].set_title(f"Corn acreage: model vs. reality (correlation = {r:.2f})")

sc = ax[1].scatter(chips.lon, chips.lat, c=chips.corn_share_pred * 100, cmap="YlOrBr", s=14, vmin=0, vmax=60)
plt.colorbar(sc, ax=ax[1], label="Predicted corn share (%)")
ax[1].set_xlabel("Longitude"); ax[1].set_ylabel("Latitude"); ax[1].set_title("Where the model finds corn (2022)")
plt.tight_layout(); plt.savefig(os.path.join(SAVE_DIR, "corn_acreage.png"), dpi=150); plt.show()
print(f"Correlation between predicted and real corn share: {r:.2f}  (1.0 = perfect)")

## Step 12 — Corn maps for the slides (the 4 chips with the most corn)
Left: what the satellite saw mid-season. Middle: the real crop map. Right: what our model predicted. **Corn is gold** so it stands out.

In [ ]:
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch

# Simple palette: corn = gold, soybeans = green, other crops = tan, nature = dark green, water = blue, built-up = gray
PALETTE = {"Natural_Vegetation": "#7a9a5a", "Forest": "#2f5d2f", "Corn": "#f2b705", "Soybeans": "#5cb85c",
           "Wetlands": "#5f9ea0", "Developed_Barren": "#9e9e9e", "Open_Water": "#2b6cb0", "Winter_Wheat": "#d9a066",
           "Alfalfa": "#b5c96a", "Fallow_Idle_Cropland": "#c8b89a", "Cotton": "#e8e1d0", "Sorghum": "#b5651d",
           "Other": "#d0d0d0"}
cmap = ListedColormap([PALETTE[c] for c in CLASS_NAMES])

def true_color(image, t=1):
    """Turn the raw satellite bands of photo t into a normal-looking color picture."""
    rgb = image[[BANDS.index("RED"), BANDS.index("GREEN"), BANDS.index("BLUE")], t].numpy().transpose(1, 2, 0)
    lo, hi = np.percentile(rgb, [2, 98])
    return np.clip((rgb - lo) / (hi - lo + 1e-6), 0, 1)

top = chips.sort_values("corn_share_true", ascending=False).head(4).index
for idx in top:
    sample = ds[idx]
    true_mask = sample["mask"].numpy().astype(float)
    true_mask[true_mask < 0] = np.nan                      # unlabeled pixels -> blank
    fig, ax = plt.subplots(1, 3, figsize=(13, 4.6))
    ax[0].imshow(true_color(sample["image"])); ax[0].set_title(f"Satellite view ({chips.date_2[idx]})")
    ax[1].imshow(true_mask, cmap=cmap, vmin=0, vmax=len(CLASS_NAMES) - 1, interpolation="nearest")
    ax[1].set_title(f"Real crops — corn {chips.corn_share_true[idx]*100:.0f}%")
    ax[2].imshow(pred_masks[idx].numpy(), cmap=cmap, vmin=0, vmax=len(CLASS_NAMES) - 1, interpolation="nearest")
    ax[2].set_title(f"Model prediction — corn {chips.corn_share_pred[idx]*100:.0f}%")
    for a in ax: a.axis("off")
    fig.legend(handles=[Patch(color=PALETTE[c], label=c.replace("_", " ")) for c in CLASS_NAMES],
               loc="lower center", ncol=7, fontsize=8, frameon=False)
    fig.suptitle(f"{chips.chip_id[idx]}  (lat {chips.lat[idx]:.2f}, lon {chips.lon[idx]:.2f})")
    plt.tight_layout(rect=(0, 0.08, 1, 0.94))
    plt.savefig(os.path.join(SAVE_DIR, f"map_{chips.chip_id[idx]}.png"), dpi=150, bbox_inches="tight")
    plt.show()

## Step 13 — Publish the results to GitHub
Uploads the results table, scores, charts, corn maps and training log to the public repo, where **Notebook 2** reads them.

**One-time setup:** create a GitHub token at *GitHub → Settings → Developer settings → Fine-grained tokens* with access to the `cornsight` repo and **Contents: Read and write**. Add it in the 🔑 **Secrets** panel (left sidebar) as `GITHUB_TOKEN`, with **Notebook access** on. The token is never printed.

In [ ]:
import base64, requests

try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except ImportError:                                  # not on Colab
    GITHUB_TOKEN = os.environ["GITHUB_TOKEN"]
headers = {"Authorization": f"Bearer {GITHUB_TOKEN}", "Accept": "application/vnd.github+json"}

def publish(local_path, repo_path):
    """Create or update one file in the GitHub repo."""
    url = f"https://api.github.com/repos/{GITHUB_REPO}/contents/{GITHUB_FOLDER}/{repo_path}"
    with open(local_path, "rb") as f:
        body = {"message": f"Training results: {repo_path}", "content": base64.b64encode(f.read()).decode()}
    existing = requests.get(url, headers=headers)
    if existing.status_code == 200:
        body["sha"] = existing.json()["sha"]          # file already there → replace it
    requests.put(url, headers=headers, json=body).raise_for_status()

log_dir = sorted(glob.glob(os.path.join(SAVE_DIR, "logs", "version_*")))[-1]   # latest training log
files = {"chip_results.csv": os.path.join(SAVE_DIR, "chip_results.csv"),        # repo path → local file
         "test_scores.csv": os.path.join(SAVE_DIR, "test_scores.csv"),
         "logs/metrics.csv": os.path.join(log_dir, "metrics.csv"),
         "logs/hparams.yaml": os.path.join(log_dir, "hparams.yaml")}
files.update({os.path.basename(p): p for p in glob.glob(os.path.join(SAVE_DIR, "*.png"))})   # charts + corn maps

for repo_path, local_path in files.items():
    publish(local_path, repo_path)
    print("  ⬆️ ", repo_path)
print(f"✅ Published {len(files)} files → https://github.com/{GITHUB_REPO}/tree/main/{GITHUB_FOLDER}")

## ✅ Done
The results are on GitHub: **https://github.com/AlonsoManchego/cornsight/tree/main/training_outputs**. Notebook 2 (`cornsight_pipeline.ipynb`) reads them from there.